# Attention Gate for U-Net

The attention gate filters an encoder skip connection before concatenating it with the decoder tensor. The decoder tensor tells the gate what region is useful, and the gate suppresses less useful skip features.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


## Basic conv block

This is the same `Conv2d -> ReLU -> Conv2d -> ReLU` block from the U-Net notebook.

In [3]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


## Attention gate

`x` is the encoder skip tensor. `g` is the decoder gating tensor. The output has the same shape as `x`, but its features are multiplied by an attention map between 0 and 1.

In [4]:
class AttentionGate(nn.Module):
    def __init__(self, skip_channels, gating_channels, hidden_channels):
        super().__init__()

        self.skip_projection = nn.Conv2d(skip_channels, hidden_channels, kernel_size=1)
        self.gating_projection = nn.Conv2d(gating_channels, hidden_channels, kernel_size=1)

        self.attention = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x, g):
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)

        x_proj = self.skip_projection(x)
        g_proj = self.gating_projection(g)

        alpha = self.attention(x_proj + g_proj)

        return x * alpha, alpha


## Shape check

For the first decoder stage in our U-Net, both tensors are spatially `64 x 48` after the decoder tensor is upsampled.

In [5]:
skip4 = torch.randn(1, 512, 64, 48)
decoder_signal = torch.randn(1, 512, 64, 48)

gate = AttentionGate(skip_channels=512, gating_channels=512, hidden_channels=256)

filtered_skip4, attention_map = gate(skip4, decoder_signal)

print("skip4:", skip4.shape)
print("decoder signal:", decoder_signal.shape)
print("attention map:", attention_map.shape)
print("filtered skip4:", filtered_skip4.shape)
print("attention min/max:", attention_map.min().item(), attention_map.max().item())


skip4: torch.Size([1, 512, 64, 48])
decoder signal: torch.Size([1, 512, 64, 48])
attention map: torch.Size([1, 1, 64, 48])
filtered skip4: torch.Size([1, 512, 64, 48])
attention min/max: 0.26632311940193176 0.7400210499763489


## Attention up block

This replaces the normal U-Net decoder pattern:

`Upsample -> concat skip -> DoubleConv`

with:

`Upsample -> attention filter skip -> concat filtered skip -> DoubleConv`

In [6]:
class AttentionUpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2,
        )

        self.attention_gate = AttentionGate(
            skip_channels=skip_channels,
            gating_channels=out_channels,
            hidden_channels=out_channels // 2,
        )

        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)

        filtered_skip, attention_map = self.attention_gate(skip, x)

        x = torch.cat([filtered_skip, x], dim=1)
        x = self.conv(x)

        return x, attention_map


## First decoder stage demo

This is the exact first upsampling stage from the U-Net we discussed.

In [7]:
latent = torch.randn(1, 1024, 32, 24)
skip4 = torch.randn(1, 512, 64, 48)

up1 = AttentionUpBlock(1024, 512, 512)

x, attention4 = up1(latent, skip4)

print("latent:", latent.shape)
print("skip4:", skip4.shape)
print("after attention up block:", x.shape)
print("attention4:", attention4.shape)


latent: torch.Size([1, 1024, 32, 24])
skip4: torch.Size([1, 512, 64, 48])
after attention up block: torch.Size([1, 512, 64, 48])
attention4: torch.Size([1, 1, 64, 48])


## Attention decoder

This is the decoder version of Attention U-Net. It returns the segmentation logits plus the four attention maps so you can inspect what each gate is focusing on.

In [8]:
class AttentionDecoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.up1 = AttentionUpBlock(1024, 512, 512)
        self.up2 = AttentionUpBlock(512, 256, 256)
        self.up3 = AttentionUpBlock(256, 128, 128)
        self.up4 = AttentionUpBlock(128, 64, 64)

        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, latent, skip1, skip2, skip3, skip4):
        x, attention4 = self.up1(latent, skip4)
        x, attention3 = self.up2(x, skip3)
        x, attention2 = self.up3(x, skip2)
        x, attention1 = self.up4(x, skip1)

        logits = self.final_conv(x)

        attention_maps = {
            "attention1": attention1,
            "attention2": attention2,
            "attention3": attention3,
            "attention4": attention4,
        }

        return logits, attention_maps


In [9]:
skip1 = torch.randn(1, 64, 512, 384)
skip2 = torch.randn(1, 128, 256, 192)
skip3 = torch.randn(1, 256, 128, 96)
skip4 = torch.randn(1, 512, 64, 48)
latent = torch.randn(1, 1024, 32, 24)

decoder = AttentionDecoder()

logits, attention_maps = decoder(latent, skip1, skip2, skip3, skip4)

print("logits:", logits.shape)

for name, attention_map in attention_maps.items():
    print(name, attention_map.shape)


logits: torch.Size([1, 1, 512, 384])
attention1 torch.Size([1, 1, 512, 384])
attention2 torch.Size([1, 1, 256, 192])
attention3 torch.Size([1, 1, 128, 96])
attention4 torch.Size([1, 1, 64, 48])
